# 01. Nibabel Intro

* ????? / ??? : ?? 1 - ????? [1]
* ???

# Contents
* Nibabel?
* Installation
* Tutorials

# Nibabel? 

#### ?? 
* [1] nibable - http://nipy.org/packages/nibabel/index.html

Nibabel aims to provide 
* read/write access to some common neuroimaging file formats.
* This package provides read +/- write access to some common medical and neuroimaging file formats, including: 
    - ANALYZE (plain, SPM99, SPM2 and later), 
    - GIFTI, 
    - NIfTI1, 
    - NIfTI2, 
    - MINC1, 
    - MINC2, 
    - MGH and 
    - ECAT as well as 
    - Philips PAR/REC. 
* We can read and write Freesurfer geometry, and read Freesurfer morphometry and annotation files. 
* There is some very limited support for DICOM. 
* NiBabel is the successor of PyNIfTI.
* The various image format classes give full or selective access to header (meta) information and access to the image data is made available via NumPy arrays.

# Installation

#### ??
* [2] nibable installation -http://nipy.org/nibabel/installation.html

In [ ]:
!pip install nibabel

# Tutorials

* Coordinate systems and affines
* Radiological vs neurological conventions
* Introduction to DICOM

#### ??
* [3] General tutorials - http://nipy.org/nibabel/tutorials.html#tutorials
* [4] Echo-planar imaging (EPI) and functional MRI - http://www.brainmapping.org/MarkCohen/Papers/EPI-fMRI.html

## Coordinate systems and affines

* Introducing Someone
* Voxel coordinates are coordinates in the image data array
* Voxel coordinates and points in space
* The scanner-subject reference space
* Voxel coordinates are in voxel space
* The affine matrix as a transformation between spaces
* The inverse of the affine gives the mapping from scanner to voxel
* The affine by example
* Other reference spaces
* Nibabel always uses an RAS+ output space

A nibabel (and nipy) image is the association of three things:
* The image data array: 
    - a 3D or 4D array of image data
* An affine array 
    - that tells you the position of the image array data in a reference space.
* image metadata 
    - (data about the data) describing the image, usually in the form of an image header.

This document describes 
* how the affine array describes the position of the image data in a reference space. 
* On the way we will define 
    - what we mean by reference space, and 
    - the reference spaces that Nibabel uses.

### Introducing Someone

#### test data
* somones_epi.nii.gz - http://nipy.org/nibabel/_downloads/someones_epi.nii.gz
* somones_anatomy.nii.gz - http://nipy.org/nibabel/_downloads/someones_anatomy.nii.gz

#### download
???(????? ?? ??? ????) ?? ???? downloads ?? ????? ???.

In [ ]:
!mkdir downloads

donwloads ??? ??? ????? ?????.

In [ ]:
!wget http://nipy.org/nibabel/_downloads/someones_epi.nii.gz -O downloads/someones_epi.nii.gz

In [ ]:
!ls *

#### We can load up the EPI image to get the image data array:

In [ ]:
import nibabel as nib

In [ ]:
epi_img = nib.load('downloads/someones_epi.nii.gz')

In [ ]:
epi_img_data = epi_img.get_data()

In [ ]:
epi_img_data.shape

#### Then we have a look at slices over the first, second and third dimensions of the array.

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def show_slices(slices):
    """ Function to display row of image slices """
    fig, axes = plt.subplots(1, len(slices))
    for i, slice in enumerate(slices):
        axes[i].imshow(slice.T, cmap="gray", origin="lower")

In [ ]:
slice_0 = epi_img_data[26, :, :]
slice_1 = epi_img_data[:, 30, :]
slice_2 = epi_img_data[:, :, 16]

In [ ]:
show_slices([slice_0, slice_1, slice_2])
plt.suptitle("Center slices for EPI image")

#### We collected an anatomical image in the same session. We can load that image and look at slices in the three axes:

In [ ]:
# ???? ?? ???(?? ???)
!wget http://nipy.org/nibabel/_downloads/someones_anatomy.nii.gz -O downloads/someones_anatomy.nii.gz

In [ ]:
anat_img = nib.load('downloads/someones_anatomy.nii.gz')
anat_img_data = anat_img.get_data()
print anat_img_data.shape

In [ ]:
show_slices([anat_img_data[28, :, :],
             anat_img_data[:, 33, :],
             anat_img_data[:, :, 28]])
plt.suptitle("Center slices for anatomical image") 

### Voxel coordinates are in voxel space

### The affine matrix as a transformation between spaces

<img src="https://i.publiclab.org/system/images/photos/000/004/507/original/geometric-transformations.jpg" width=600 />

<img src="https://www.cs.auckland.ac.nz/courses/compsci773s1c/lectures/ImageProcessing-html/affine-steps.gif" width=600 />

<img src="http://image.slidesharecdn.com/matrix2d-131120051740-phpapp02/95/matrix-2-d-24-638.jpg?cb=1386427966" width=600 />

<img src="http://i.stack.imgur.com/roKQ8.png" width=600 />

### The inverse of the affine gives the mapping from scanner to voxel

The affine arrays we have described so far have another pleasant property ? they are usually invertible.

### The affine by example

In [ ]:
# Set numpy to print 3 decimal points and suppress small values
import numpy as np
np.set_printoptions(precision=3, suppress=True)
# Print the affine
print epi_img.affine

As you see, the last row is [0,0,0,1]

### Applying the affine

In [ ]:
# To make the affine simpler to apply, we split it into MM and (a,b,c)(a,b,c):
M = epi_img.affine[:3, :3]
abc = epi_img.affine[:3, 3]

In [ ]:
# Then we can define our function f:
def f(i, j, k):
    """ Return X, Y, Z coordinates for i, j, k """
    return M.dot([i, j, k]) + abc

In [ ]:
# The labels on the localizer image give the impression 
# that the center voxel of someones_epi.nii.gz was 
# a little above the magnet isocenter. Now we can check:
epi_vox_center = (np.array(epi_img_data.shape) - 1) / 2.
print epi_vox_center
print f(epi_vox_center[0], epi_vox_center[1], epi_vox_center[2])

That means the center of the image field of view is at the isocenter of the magnet on the left to right axis, and is around 4.2mm posterior to the isocenter and ~8.5 mm above the isocenter.

The parameters in the affine array can therefore give the position of any voxel coordinate, relative to the scanner RAS+ reference space.

We get the same result from applying the affine directly instead of using MM and (a,b,c)(a,b,c) in our function. As above, we need to add a 1 to the end of the vector to apply the 4 by 4 affine matrix.

In [ ]:
epi_img.affine.dot(list(epi_vox_center) + [1])

#### nibable's apply_affine

In fact nibabel has a function apply_affine that applies an affine to an (i,j,k)(i,j,k) point by splitting the affine into MM and abcabc then multiplying and adding as above:

In [ ]:
from nibabel.affines import apply_affine
apply_affine(epi_img.affine, epi_vox_center)

Now we can apply the affine, we can use matrix inversion on the anatomical affine to map between voxels in the EPI image and voxels in the anatomical image.

In [ ]:
import numpy.linalg as npl
epi_vox2anat_vox = npl.inv(anat_img.affine).dot(epi_img.affine)

In [ ]:
print epi_vox2anat_vox

What is the voxel coordinate in the anatomical corresponding to the voxel center of the EPI image?

In [ ]:
apply_affine(epi_vox2anat_vox, epi_vox_center)

The voxel coordinate of the center voxel of the anatomical image is:

In [ ]:
anat_vox_center = (np.array(anat_img_data.shape) - 1) / 2.
anat_vox_center

### The affine as a series of transformations

You can think of the image affine as a combination of a series of transformations to go from voxel coordinates to mm coordinates in terms of the magnet isocenter. Here is the EPI affine broken down into a series of transformations, with the results shown on the localizer image:

<img src="http://nipy.org/nibabel/_images/illustrating_affine.png" width=600 />

We start by putting the voxel grid onto the isocenter coordinate system, so a translation of one voxel equates to a translation of one millimeter in the isocenter coordinate system. Our EPI image would then have the black bounding box in the image above. Next we scale the voxels to millimeters by scaling by the voxel size (green bounding box). We could do this with an affine:

In [ ]:
scaling_affine = np.array([[3, 0, 0, 0],
                            [0, 3, 0, 0],
                            [0, 0, 3, 0],
                            [0, 0, 0, 1]])

After applying this affine, when we move one voxel in any direction, we are moving 3 millimeters in that direction:

In [ ]:
one_vox_axis_0 = [1, 0, 0]
apply_affine(scaling_affine, one_vox_axis_0)

Next we rotate the scaled voxels around the first axis by 0.3 radians

In [ ]:
cos_gamma = np.cos(0.3)
sin_gamma = np.sin(0.3)
rotation_affine = np.array([[1, 0, 0, 0],
                            [0, cos_gamma, -sin_gamma, 0],
                            [0, sin_gamma, cos_gamma, 0],
                            [0, 0, 0, 1]])
affine_so_far = rotation_affine.dot(scaling_affine)
print affine_so_far

The EPI voxel block coordinates transformed by affine_so_far are at the position of the yellow box on the figure.

<img src="http://nipy.org/nibabel/_images/illustrating_affine.png" width=600 />

Finally we translate the 0, 0, 0 coordinate at the bottom, posterior, left corner of our array to be at its final position relative to the isocenter, which is -78, -76, -64:

In [ ]:
translation_affine = np.array([[1, 0, 0, -78],
                                [0, 1, 0, -76],
                                [0, 0, 1, -64],
                                [0, 0, 0, 1]])
whole_affine = translation_affine.dot(affine_so_far)
print whole_affine

#### ????
* [1] nibable - http://nipy.org/packages/nibabel/index.html
* [2] nibable installation -http://nipy.org/nibabel/installation.html
* [3] General tutorials - http://nipy.org/nibabel/tutorials.html#tutorials
* [4] Echo-planar imaging (EPI) and functional MRI - http://www.brainmapping.org/MarkCohen/Papers/EPI-fMRI.html